### Building A RAG system with langchain and Chromadb

In [1]:
import os

In [4]:
#langchain import 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import Chroma

import numpy as np
from typing import List


#### Creating Sample Docs

In [ ]:
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine Learning (ML) is a subset of Artificial Intelligence that enables
    computers to learn patterns from data without being explicitly programmed.
    It is widely used in applications such as recommendation systems, spam
    detection, fraud detection, predictive analytics, and image classification.
    Machine Learning algorithms can be supervised, unsupervised, or reinforcement
    learning based depending on the type of training data available.
    """,

    """
    Deep Learning Fundamentals
    
    Deep Learning (DL) is a specialized branch of Machine Learning that uses
    artificial neural networks with multiple hidden layers to learn complex
    patterns from large amounts of data. Deep Learning has achieved remarkable
    success in computer vision, speech recognition, autonomous vehicles, and
    medical image analysis. Popular frameworks for Deep Learning include
    TensorFlow and PyTorch.
    """,

    """
    NLP Fundamentals 
         
    Natural Language Processing (NLP) is a field of Artificial Intelligence
    that focuses on enabling computers to understand, interpret, and generate
    human language. NLP powers applications such as chatbots, language
    translation, sentiment analysis, text summarization, question answering,
    and virtual assistants like ChatGPT. Modern NLP systems often use
    Transformer-based models such as BERT and GPT.
    """
]

print(sample_docs[0])

In [8]:
### save sample doc to files
import tempfile
temp_dir = tempfile.mkdtemp()
for i,doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)



print(f"sample Docs Created in {temp_dir}")        
        

sample Docs Created in C:\Users\mrraj\AppData\Local\Temp\tmprobegqug


### Document Loading


In [ ]:
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader(
    temp_dir,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'}
)

documents = loader.load()
print(f"Loaded {len(documents)} documents")
print(f"\n First Document Preview:")
print(documents[0].page_content[:200] + "...")

### Document SPlitting


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len,
    separators=[" "] 
)

chunks =splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f" Content :{chunks[0].page_content[:150]}")
print(f"Metadat:{chunks[0].metadata}")




### Embedding MOdel

In [18]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
  model_name = "sentence-transformers/all-miniLM-L6-V2"
)
embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='sentence-transformers/all-miniLM-L6-V2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
# texts = [doc.page_content for doc in documents]

# vectors = embeddings.embed_documents(texts)

# print(f"Total Vectors : {len(vectors)}")
# print(f"Dimension     : {len(vectors[0])}")
# print(texts[0])
# print(vectors)


### Initialize the chromaDB vector and the store the chunks in vector representation  

In [36]:
presist_directory = "./chroma_db"
vectorestore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=presist_directory,
    collection_name="Rag_collection"
) 

### Testing Similarity Search

In [41]:
query = "What is Machine Learning?"

results = vectorestore.similarity_search(query, k=3)
results[0].page_content


'Machine Learning Fundamentals\n\n    Machine Learning (ML) is a subset of Artificial Intelligence that enables\n    computers to learn patterns from data without being explicitly programmed.\n    It is widely used in applications such as recommendation systems, spam\n    detection, fraud detection, predictive analytics, and image classification.\n    Machine Learning algorithms can be supervised, unsupervised, or reinforcement\n    learning based depending on the type of training data available.'

In [42]:
query = "What is NLP?"

results = vectorestore.similarity_search(query, k=3)
results[0].page_content


'NLP Fundamentals \n\n    Natural Language Processing (NLP) is a field of Artificial Intelligence\n    that focuses on enabling computers to understand, interpret, and generate\n    human language. NLP powers applications such as chatbots, language\n    translation, sentiment analysis, text summarization, question answering,\n    and virtual assistants like ChatGPT. Modern NLP systems often use\n    Transformer-based models such as BERT and GPT.'

### Advance similarity search with score

In [46]:
query = "What is Machine Learning?"
res = vectorestore.similarity_search_with_score(query,k=3)
for doc, score in res:
    print("=" * 60)
    print("Score :", score)
    print(doc.page_content)


Score : 0.46563035249710083
Machine Learning Fundamentals

    Machine Learning (ML) is a subset of Artificial Intelligence that enables
    computers to learn patterns from data without being explicitly programmed.
    It is widely used in applications such as recommendation systems, spam
    detection, fraud detection, predictive analytics, and image classification.
    Machine Learning algorithms can be supervised, unsupervised, or reinforcement
    learning based depending on the type of training data available.
Score : 0.46563035249710083
Machine Learning Fundamentals

    Machine Learning (ML) is a subset of Artificial Intelligence that enables
    computers to learn patterns from data without being explicitly programmed.
    It is widely used in applications such as recommendation systems, spam
    detection, fraud detection, predictive analytics, and image classification.
    Machine Learning algorithms can be supervised, unsupervised, or reinforcement
    learning based depend

### Inlitialize our LLM

In [ ]:
from dotenv import load_dotenv
load_dotenv()
from langchain_google_genai import ChatGoogleGenerativeAI

Ab8RN6K2MH9MmAnxRHD9Swv521vmLJidPVYzFyNNjrTQfkUQJg
50


In [24]:

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)
response = llm.invoke("what is capital of india ")
print(response.content)

ChatGoogleGenerativeAIError: Error calling model 'gemini-3.6-flash' (INVALID_ARGUMENT): 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}